# 9-2절 연습 문제 풀이

이 노트북은 9-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch09/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 9-2/9-3절은 본문 예제(09-02, 09-03 노트북)의 코드를 재사용한다.
# 아래 도우미는 정렬 과제(연습 9-9, 9-11)에서 공통으로 사용한다.
import random as _random
from torch.utils.data import Dataset, DataLoader
PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

def make_sort_pairs(n=5000, count=10, seed=SEED):
    rng = _random.Random(seed)
    pairs = []
    for _ in range(n):
        nums = [rng.randint(1, 1000) for _ in range(count)]
        pairs.append((', '.join(map(str, nums)),
                      ', '.join(map(str, sorted(nums)))))
    return pairs

def build_vocab(texts):
    chars = sorted({c for t in texts for c in t})
    tokens = [PAD, SOS, EOS, UNK] + chars
    return {t: i for i, t in enumerate(tokens)}

class SeqDataset(Dataset):
    def __init__(self, pairs, sv, tv, sl=60, tl=62):
        self.pairs, self.sv, self.tv, self.sl, self.tl = pairs, sv, tv, sl, tl
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        s, t = self.pairs[i]
        src = [self.sv.get(c, self.sv[UNK]) for c in s][:self.sl]
        src += [self.sv[PAD]] * (self.sl - len(src))
        tgt = [self.tv[SOS]] + [self.tv.get(c, self.tv[UNK]) for c in t] + [self.tv[EOS]]
        tgt = tgt[:self.tl] + [self.tv[PAD]] * (self.tl - len(tgt))
        return torch.tensor(src), torch.tensor(tgt)

## 연습 9-7

[코드 9-15]와 [코드 9-16]은 학습 함수에서 길이 정보 텐서를 계산해 모델에 전달한다. 본문에서 함께 소개한 '데이터셋에서 계산한 길이를 배치 병합 함수에서 취합하는 방식'과 '배치 병합 함수에서 길이를 계산하고 취합하는 방식'으로도 구현해 보자. 구현 후 세 가지 방식의 장단점을 정리해 보자.

### 풀이

길이 정보를 만드는 위치에 따라 세 가지 방식이 있다.

| 방식 | 구현 위치 | 장점 | 단점 |
|---|---|---|---|
| 학습 함수에서 계산 | `(src != PAD).sum(dim=1)` | 데이터셋·병합 함수를 건드리지 않아 간단 | 학습·검증·생성 함수마다 같은 코드를 반복 |
| 데이터셋에서 계산 → 병합 함수가 취합 | `__getitem__`이 길이도 반환 | 샘플 하나의 성질이므로 책임이 명확, 재사용 쉬움 | 데이터셋 반환값이 늘어 병합 함수도 함께 바꿔야 함 |
| 병합 함수에서 계산 | `collate_fn` 안에서 계산 | 패딩과 길이 계산이 한곳에 모임 | 병합 함수를 직접 작성해야 함 |

**패딩을 병합 함수에서 수행한다면 세 번째 방식**이 가장 자연스럽다. 패딩 전 원래 길이를 그 자리에서 알 수 있기 때문이다. 데이터셋에서 이미 고정 길이로 패딩한다면 첫 번째가 간단하다.

In [ ]:
# 방식 3) 병합 함수에서 패딩과 길이 계산을 함께 처리
def collate_with_length(batch, pad_idx=0):
    srcs, tgts = zip(*batch)
    lengths = torch.tensor([(s != pad_idx).sum() for s in srcs])
    return torch.stack(srcs), torch.stack(tgts), lengths

pairs = make_sort_pairs(200)
sv = tv = build_vocab([s for s, _ in pairs] + [t for _, t in pairs])
loader = DataLoader(SeqDataset(pairs, sv, tv), batch_size=8,
                    collate_fn=collate_with_length)
src, tgt, lengths = next(iter(loader))
print(f'입력 {tuple(src.shape)}, 정답 {tuple(tgt.shape)}, 길이 {lengths.tolist()}')

## 연습 9-8

인코더에서 패킹을 적용하지 않고(<pad> 토큰을 그대로 둔 채) 학습한 결과를, 패킹을 적용한 경우와 학습 과정 및 모델 성능 면에서 비교해 보자. 차이를 확인할 수 있다면 그 차이가 어디에서 기인한 것인지도 함께 정리해 보자.

### 풀이

**패킹을 하지 않으면** 인코더 LSTM이 `<pad>` 토큰까지 순차적으로 처리한다. 그 결과 두 가지 문제가 생긴다.

1. **마지막 숨겨진 상태가 오염된다.** LSTM이 반환하는 `h`는 마지막 시점의 상태인데, 짧은 입력은 뒤쪽이 전부 `<pad>`라 '패딩만 잔뜩 본 상태'가 콘텍스트 벡터가 된다. 정작 중요한 정보는 그 앞에서 희석된다.
2. **불필요한 연산이 늘어난다.** 의미 없는 토큰에 대해서도 LSTM 계산을 수행한다.

**패킹을 적용하면** 각 샘플의 실제 길이만큼만 계산하고, 마지막 유효 토큰의 상태를 `h`로 돌려준다. 그래서 학습이 더 빠르고 콘텍스트 벡터의 품질이 좋아진다.

다만 `padding_idx`로 `<pad>` 임베딩을 0으로 고정해 두면 영향이 줄어, 입력 길이 편차가 작은 이 예제에서는 성능 차이가 크지 않을 수 있다. 길이 편차가 큰 데이터일수록 패킹의 효과가 뚜렷하다.

## 연습 9-9

1에서 1,000 사이의 정수 10개를 무작위로 뽑아 쉼표로 구분한 문자열(예: 5, 724, 223, 695, ...)을 입력하면, 오름차순으로 정렬한 문자열(예: 5, 223, 695, 724, ...)을 생성하는 Seq2Seq 모델을 만들어 보자. 이번 절부터 10-3절까지 절마다 정렬과 관련된 연습 문제가 있으니, 10-3절까지 학습한 후 9장과 10장 전체를 복습한다는 느낌으로 함께 풀어 보는 것도 좋은 선택이다.

In [ ]:
pairs = make_sort_pairs(5000)
sv = tv = build_vocab([s for s, _ in pairs] + [t for _, t in pairs])
rev = {i: t for t, i in tv.items()}
loader = DataLoader(SeqDataset(pairs, sv, tv), batch_size=64, shuffle=True)
print(f'예: {pairs[0][0][:40]} ...\n  -> {pairs[0][1][:40]} ...')
print(f'어휘 {len(sv)}개')

class Seq2Seq(nn.Module):
    def __init__(self, sv_size, tv_size, embed=32, hidden=256):
        super().__init__()
        self.enc_emb = nn.Embedding(sv_size, embed, padding_idx=0)
        self.enc = nn.LSTM(embed, hidden, batch_first=True)
        self.dec_emb = nn.Embedding(tv_size, embed, padding_idx=0)
        self.dec = nn.LSTM(embed, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, tv_size)
    def forward(self, src, tgt, forcing=0.5):
        _, (h, c) = self.enc(self.enc_emb(src))
        token, outs = tgt[:, :1], []
        for t in range(1, tgt.size(1)):
            out, (h, c) = self.dec(self.dec_emb(token), (h, c))
            logits = self.fc(out.squeeze(1)); outs.append(logits.unsqueeze(1))
            token = (tgt[:, t:t+1] if torch.rand(1).item() < forcing
                     else logits.argmax(1, keepdim=True))
        return torch.cat(outs, 1)

torch.manual_seed(SEED)
model = Seq2Seq(len(sv), len(tv)).to(device)
crit = nn.CrossEntropyLoss(ignore_index=0)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for e in range(1, 21):
    model.train(); tot = n = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        out = model(src, tgt)
        loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); n += 1
    if e % 5 == 0: print(f'{e}/20 손실 {tot / n:.4f}')

정렬은 날짜 변환보다 훨씬 어렵다. **출력의 모든 위치가 입력 전체에 의존**하기 때문이다. 콘텍스트 벡터 하나에 10개 숫자의 순서 정보를 모두 담아야 하므로 정보 병목이 심하다.

이 한계가 다음 절 어텐션([연습 문제 9-11])과 10장 트랜스포머([연습 문제 10-1])로 이어지는 자연스러운 동기가 된다.

## 연습 9-10

[도전 문제] 데이터로더와 모델의 수정에 맞춰 본문 예제에서 다룬 학습 함수뿐 아니라 검증 함수와 생성 함수도 수정해야 한다. 깃허브 예제 노트북의 검증 함수와 생성 함수를 참고하지 말고 직접 구현해 보자. 검증 함수는 검증 손실과 정확도(순차 데이터 단위의 정확도, 모든 토큰이 일치할 때만 정답)를 계산하고, 생성 함수는 날짜 문자열을 입력받아 출력 형식으로 변환해 생성하도록 만든다. 손실과 정확도 계산에 <pad>가 반영되지 않도록 주의하자.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, tgt_vocab):
    """검증 손실과 순차 데이터 단위 정확도(모든 토큰 일치)를 계산"""
    model.eval()
    crit = nn.CrossEntropyLoss(ignore_index=tgt_vocab[PAD])
    tot = n = correct = total = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        out = model(src, tgt, forcing=0.0)          # 검증에서는 교사 강제 없음
        tot += crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1)).item(); n += 1
        pred = out.argmax(-1)
        mask = tgt[:, 1:] != tgt_vocab[PAD]
        match = ((pred == tgt[:, 1:]) | ~mask).all(dim=1)   # 유효 토큰이 모두 일치
        correct += match.sum().item(); total += len(match)
    return tot / n, correct / total * 100

@torch.no_grad()
def generate(model, text, sv, tv, rev, max_len=62):
    model.eval()
    src = [sv.get(c, sv[UNK]) for c in text][:60]
    src += [sv[PAD]] * (60 - len(src))
    src = torch.tensor([src], device=device)
    _, (h, c) = model.enc(model.enc_emb(src))
    token, out = torch.tensor([[tv[SOS]]], device=device), []
    for _ in range(max_len):
        o, (h, c) = model.dec(model.dec_emb(token), (h, c))
        nxt = model.fc(o.squeeze(1)).argmax(1, keepdim=True)
        if nxt.item() == tv[EOS]: break
        out.append(rev[nxt.item()]); token = nxt
    return ''.join(out)

vl, acc = evaluate(model, loader, tv)
print(f'검증 손실 {vl:.4f} / 순차 정확도 {acc:.2f}%')
print(f'입력: {pairs[0][0]}')
print(f'생성: {generate(model, pairs[0][0], sv, tv, rev)}')
print(f'정답: {pairs[0][1]}')

검증 함수의 핵심은 두 가지다. ① **교사 강제를 끄고**(`forcing=0.0`) 실제 추론과 같은 조건으로 평가한다. ② 순차 데이터 정확도는 `<pad>`를 제외한 **모든 토큰이 일치할 때만** 정답으로 센다.

생성 함수는 `<sos>`에서 시작해 `<eos>`가 나오거나 최대 길이에 이를 때까지 한 토큰씩 이어 붙인다.